# TriFuseRec — Amazon Beauty (Colab)

Second-dataset run of the full TriFuseRec pipeline, matching ML-1M:
1. download Amazon Beauty (2014, 5-core) and preprocess (chronological leave-two-out, random same-day tie-break)
2. train **MostPopular, DeepFM, SASRec, NGCF** for each seed (early stopping on validation)
3. **gating comparison** (`src/cand_gate.py`): best single, equal weights, fixed weights, per-user gate, candidate-aware gate
4. analysis + report (`src/analyze.py`, `src/report.py`)

**Runtime → Change runtime type → T4 GPU.** Standard RAM (12.7 GB) is enough; the gate peaks around 6 GB.

Code, data and results live in Google Drive; the large score matrices (~1 GB each) stay on Colab's local disk
and are deleted after each seed's gate run. Re-running section 5 resumes: finished runs are skipped.

## 1. Settings

In [ ]:
USE_DRIVE = True          # keep code + results in Google Drive (recommended; survives disconnects)
CODE_SOURCE = 'upload'    # 'upload' -> upload trifuserec_code.zip ; 'git' -> clone GIT_URL
GIT_URL = ''              # e.g. 'https://github.com/<you>/TriFuseRec.git' (if the code is pushed)
REFRESH_CODE = True       # replace src/, scripts/, tests/ with the uploaded version
FRESH_RUN = True          # move old Beauty data/results to a dated backup folder and start clean
                          # (set False when resuming an interrupted run of THIS version)
SEEDS = '42 43 44'

## 2. GPU check and working folder

In [ ]:
import os, shutil, time, torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - switch runtime to GPU')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/TriFuseRec'
else:
    WORK = '/content/TriFuseRec'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('working folder:', WORK)

## 3. Get the code
Upload **`trifuserec_code.zip`** (from the repo's `colab/` folder) when prompted — not a single `.py` file.

In [ ]:
os.chdir(WORK)
if REFRESH_CODE:
    for d in ('src', 'scripts', 'tests'):
        shutil.rmtree(d, ignore_errors=True)
if not os.path.exists('src/train.py'):
    if CODE_SOURCE == 'git':
        !git clone {GIT_URL} /content/_repo && cp -r /content/_repo/. {WORK}/
    else:
        from google.colab import files
        print('Choose trifuserec_code.zip')
        up = files.upload()
        name = next(iter(up))
        if not name.endswith('.zip'):
            os.remove(name)
            raise ValueError(f'You uploaded {name}. Re-run this cell and upload trifuserec_code.zip')
        !unzip -o -q "{name}" -d {WORK} && rm "{name}"
assert os.path.exists('src/cand_gate.py') and os.path.exists('scripts/run_beauty.sh'), 'code not unpacked - re-run this cell'
print('code OK:', sorted(f for f in os.listdir('src') if f.endswith('.py')))

## 4. Download Amazon Beauty (≈145 MB, skipped if present) and preprocess

In [ ]:
os.chdir(WORK)
if FRESH_RUN:
    stamp = time.strftime('%Y%m%d-%H%M')
    for d in ('data/processed_beauty', 'results/beauty'):
        if os.path.exists(d):
            shutil.move(d, f'{d}_old_{stamp}'); print('moved', d, '->', f'{d}_old_{stamp}')
os.makedirs('data/raw/amazon-beauty', exist_ok=True)
base = 'http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/'
for f in ['reviews_Beauty_5.json.gz', 'meta_Beauty.json.gz']:
    if not os.path.exists(f'data/raw/amazon-beauty/{f}'):
        !wget -q --show-progress -O data/raw/amazon-beauty/{f} {base}{f}
if not os.path.exists('data/processed_beauty/train.csv'):
    !python -m src.preprocess_beauty
!head -2 data/processed_beauty/train.csv
!cat data/processed_beauty/stats.json

## 5. Train all models + gating for every seed, then analysis and report
Progress prints every ~30 s; the full log is `results/beauty_run.log`. Expect roughly 1–2 h on a T4.
If Colab disconnects: re-run sections 2–3 (with `REFRESH_CODE = False`), skip 4, and re-run this cell with `FRESH_RUN = False`.

In [ ]:
import re, subprocess
os.chdir(WORK); os.makedirs('results', exist_ok=True)
env = f'PY=python SEEDS="{SEEDS}" SCORES_DIR=/content/beauty_scores CLEAN_SCORES=1'
os.system(f'{env} bash scripts/run_beauty.sh > results/beauty_run.log 2>&1 &')
seen = 0
while True:
    time.sleep(30)
    lines = open('results/beauty_run.log').read().splitlines()
    for line in lines[seen:]:
        if re.search(r'epoch|early|VALIDATION|TEST|protocol=|best_single|equal_weights|fixed_weights|per_user_gate|candidate_gate|skip|re-scoring|ALL_DONE|Error|Traceback|Killed', line):
            print(line, flush=True)
    seen = len(lines)
    if not subprocess.run(['pgrep', '-f', 'run_beauty.sh'], capture_output=True).stdout.strip():
        break
if not any('ALL_DONE' in l for l in lines):
    print('\n*** RUN STOPPED BEFORE FINISHING - last 40 log lines: ***')
    print('\n'.join(lines[-40:]))
    raise RuntimeError('run_beauty.sh failed; see the log lines above')
print('finished OK')

## 6. Results

In [ ]:
import glob, pandas as pd
from IPython.display import Markdown
os.chdir(WORK)
pd.set_option('display.float_format', '{:.4f}'.format)
if not os.path.exists('results/beauty/analysis/test_summary.csv'):
    raise SystemExit('No analysis yet - section 5 has not finished. Re-run it; finished runs are skipped.')
print('===== Baselines, test: mean / std over seeds =====')
display(pd.read_csv('results/beauty/analysis/test_summary.csv', header=[0, 1], index_col=0))
g = pd.read_csv('results/beauty/analysis/test_groups_summary.csv')
print('===== Baselines, test NDCG@10 by user history length / target-item popularity =====')
display(g.pivot_table(index=['group_type', 'group'], columns='model', values='NDCG@10'))
gm = pd.concat([pd.read_csv(p).assign(seed=p.split('seed')[-1].split('/')[0])
                for p in sorted(glob.glob('results/beauty/cand_gate/seed*/metrics.csv'))])
print('===== Gating comparison, test: mean / std over seeds =====')
display(gm[gm.split == 'test'].groupby('method', sort=False)[['Recall@10', 'NDCG@10', 'Recall@20', 'NDCG@20']].agg(['mean', 'std']))
w = [c for c in gm.columns if c.startswith('w_')]
print('===== Average expert weights (test) =====')
display(gm[gm.split == 'test'].groupby('method', sort=False)[w].mean())
rep = sorted(glob.glob('results/beauty/analysis/report_beauty*/report.md'))
if rep:
    print('Full report:', rep[-1]); display(Markdown(open(rep[-1]).read()))

## 7. Download results
(Includes model checkpoints and gate weights; score matrices are not kept.)

In [ ]:
!cd {WORK} && zip -q -r /content/beauty_results.zip results/beauty data/processed_beauty/stats.json -x "results/beauty/scores/*"
from google.colab import files
files.download('/content/beauty_results.zip')